In [1]:
#第11章/加载数据集
from datasets import load_dataset
import torchvision
import torch


def get_dataset():
    #加载数据集
    dataset = load_dataset('lansinuote/gen.5.flower.book', split='train')

    #删除多余的字段
    dataset = dataset.remove_columns(['cls'])

    #图像数据预处理
    compose = torchvision.transforms.Compose([
        torchvision.transforms.Resize(64),
        torchvision.transforms.ToTensor(),
        lambda x: x * 2 - 1,
    ])

    def f(data):
        image = compose(data['image'][0]).unsqueeze(dim=0)
        return {'image': image}

    dataset = dataset.with_transform(f)

    #为了加速数据遍历的效率,把全体数据载入内存以加速IO
    dataset_tensor = torch.empty(len(dataset), 3, 64, 64)

    for i in range(len(dataset)):
        dataset_tensor[i] = dataset[i]['image']

    return dataset_tensor


dataset = get_dataset()

dataset.shape, dataset.dtype

Using custom data configuration lansinuote--gen.5.flower.book-34a531175c385260
Found cached dataset parquet (/root/.cache/huggingface/datasets/lansinuote___parquet/lansinuote--gen.5.flower.book-34a531175c385260/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec)


(torch.Size([2000, 3, 64, 64]), torch.float32)

In [2]:
#第11章/定义loader
loader = torch.utils.data.DataLoader(dataset=dataset,
                                     batch_size=64,
                                     shuffle=True,
                                     drop_last=True)

len(loader), next(iter(loader)).shape

(31, torch.Size([64, 3, 64, 64]))

In [3]:
#第11章/show函数
def show(images):
    from matplotlib import pyplot as plt

    images = images.to('cpu').detach()[:50]
    images = images.permute(0, 2, 3, 1)
    images = (images + 1) / 2

    plt.figure(figsize=(20, 10))

    for i in range(len(images)):
        plt.subplot(5, 10, i + 1)
        plt.imshow(images[i])
        plt.axis('off')

    plt.show()


show(next(iter(loader)))

<Figure size 2000x1000 with 50 Axes>

In [4]:
#第11章/定义CLS模型
cls = torch.nn.Sequential(
    torch.nn.Conv2d(3, 64, kernel_size=5, stride=2, padding=1),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Conv2d(64, 128, kernel_size=5, stride=2, padding=1),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Conv2d(128, 256, kernel_size=5, stride=2, padding=1),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Conv2d(256, 512, kernel_size=5, stride=2, padding=1),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=0),
    torch.nn.LeakyReLU(0.2),
    torch.nn.Flatten(),
    torch.nn.Linear(512, 1),
)

cls(torch.randn(2, 3, 64, 64)).shape

torch.Size([2, 1])

In [5]:
#第11章/定义GEN模型
class Block(torch.nn.Module):

    def __init__(self, dim_in, dim_out):
        super().__init__()

        def block(dim_in, dim_out, kernel_size=3, stride=1, padding=1):
            return (
                torch.nn.ConvTranspose2d(dim_in,
                                         dim_out,
                                         kernel_size=kernel_size,
                                         stride=stride,
                                         padding=padding),
                torch.nn.BatchNorm2d(dim_out),
                torch.nn.LeakyReLU(),
            )

        self.s = torch.nn.Sequential(
            *block(dim_in, dim_in),
            *block(dim_in, dim_in),
            *block(dim_in, dim_in),
            *block(dim_in, dim_out, kernel_size=3, stride=2, padding=0),
            *block(dim_out, dim_out),
            *block(dim_out, dim_out),
            *block(dim_out, dim_out),
        )

        self.res = torch.nn.ConvTranspose2d(dim_in,
                                            dim_out,
                                            kernel_size=3,
                                            stride=2,
                                            padding=0)

    def forward(self, x):
        return self.s(x) + self.res(x)


gen = torch.nn.Sequential(
    torch.nn.Linear(128, 256 * 4 * 4),
    torch.nn.InstanceNorm1d(256 * 4 * 4),
    torch.nn.Unflatten(dim=1, unflattened_size=(256, 4, 4)),
    Block(256, 128),
    Block(128, 64),
    Block(64, 32),
    Block(32, 3),
    torch.nn.UpsamplingNearest2d(size=64),
    torch.nn.Conv2d(in_channels=3,
                    out_channels=3,
                    kernel_size=1,
                    stride=1,
                    padding=0),
    torch.nn.Tanh(),
)

gen(torch.randn(2, 128)).shape

torch.Size([2, 3, 64, 64])

In [6]:
#第11章/初始化工具类
def set_requires_grad(model, requires_grad):
    for param in model.parameters():
        param.requires_grad_(requires_grad)


def wasserstein(pred, label):
    return -(pred * label).mean()


optimizer_cls = torch.optim.Adam(cls.parameters(), lr=2e-4)
optimizer_gen = torch.optim.Adam(gen.parameters(), lr=2e-4)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

cls.to(device)
gen.to(device)

cls.train()
gen.train()

device

'cuda'

In [7]:
#第11章/计算梯度为loss的函数
def get_gradient_penalty(real, fake):
    #real -> [64, 3, 64, 64]
    #fake -> [64, 3, 64, 64]

    r = torch.rand((64, 1, 1, 1), device=device)
    r.requires_grad = True

    #[64, 3, 64, 64]
    merge = r * real + (1 - r) * fake

    #[64, 3, 64, 64] -> [64, 1]
    pred = cls(merge)

    grad = torch.autograd.grad(inputs=merge,
                               outputs=pred,
                               grad_outputs=torch.ones(64, 1, device=device),
                               create_graph=True,
                               retain_graph=True)

    #[64, 3, 64, 64] -> [64, 12288]
    grad = grad[0].reshape(64, -1)

    #[64, 12288] -> [64]
    grad = grad.norm(p=2, dim=1)

    #[64] -> scala
    return (1 - grad).pow(2).mean()


get_gradient_penalty(torch.randn(64, 3, 64, 64, device=device),
                     torch.randn(64, 3, 64, 64, device=device))

tensor(0.9880, device='cuda:0', grad_fn=<MeanBackward0>)

In [8]:
#第11章/训练CLS模型的函数
def train_cls():
    set_requires_grad(cls, True)
    set_requires_grad(gen, False)

    #得到三份数据
    data_real = next(iter(loader)).to(device)
    with torch.no_grad():
        data_fake = gen(torch.randn(64, 128, device=device))

    #分别计算
    pred_real = cls(data_real)
    pred_fake = cls(data_fake)

    #求loss,加权求和
    loss_real = wasserstein(pred_real, torch.ones(64, 1, device=device))
    loss_fake = wasserstein(pred_fake, -torch.ones(64, 1, device=device))
    loss_grad = get_gradient_penalty(data_real, data_fake)

    loss = loss_fake + loss_real + loss_grad * 10

    loss.backward()
    optimizer_cls.step()
    optimizer_cls.zero_grad()

    return loss.item()


train_cls()

9.878754615783691

In [9]:
#第11章/训练GEN模型的函数
def train_gen():
    set_requires_grad(cls, False)
    set_requires_grad(gen, True)

    pred = cls(gen(torch.randn(64, 128, device=device)))

    loss = wasserstein(pred, torch.ones(64, 1, device=device))
    loss.backward()
    optimizer_gen.step()
    optimizer_gen.zero_grad()

    return loss.item()


train_gen()

0.08758492767810822

In [10]:
#第11章/训练
def train():
    for epoch in range(20_0000):
        for _ in range(5):
            loss_cls = train_cls()

        loss_gen = train_gen()

        if epoch % 1_0000 == 0:
            print(epoch, loss_cls, loss_gen)
            with torch.no_grad():
                pred = gen(torch.randn(10, 128, device=device))
            show(pred)
            
    torch.save(gen.to('cpu'), 'save/gen.model')
    torch.save(cls.to('cpu'), 'save/cls.model')


train()

0 -0.7289295196533203 11.508230209350586


<Figure size 2000x1000 with 10 Axes>

10000 -17.69113540649414 8.688091278076172


<Figure size 2000x1000 with 10 Axes>

20000 -24.43415641784668 8.451485633850098


...

<Figure size 2000x1000 with 10 Axes>

180000 -29.20619010925293 9.78554916381836


<Figure size 2000x1000 with 10 Axes>

190000 -27.564098358154297 9.468210220336914


<Figure size 2000x1000 with 10 Axes>

In [11]:
#第11章/测试
gen = torch.load('save/gen.model')

with torch.no_grad():
    pred = gen(torch.randn(50, 128))

show(pred)

<Figure size 2000x1000 with 50 Axes>

In [12]:
#第11章/在线加载笔者训练好的模型并测试
from transformers import PreTrainedModel, PretrainedConfig


class Model(PreTrainedModel):
    config_class = PretrainedConfig

    def __init__(self, config):
        super().__init__(config)
        self.cls = cls.to('cpu')
        self.gen = gen.to('cpu')


#加载训练好的模型
gen = Model.from_pretrained('lansinuote/gen.5.wgangp.book').gen

with torch.no_grad():
    pred = gen(torch.randn(50, 128))

show(pred)

<Figure size 2000x1000 with 50 Axes>